# Tarea entrega final: Fine-tunning Stable Difusion
## 1. Datos generales de la tarea
- De acuerdo al documento PDF, he hecho las siguientes implementaciones acciones:
    - Crear un perfil en huggingFace (https://huggingface.co/Emmy1993)  
    - Exponer ahí mis dos modelos finetuneados (Más adelante explicaré porque tengo dos en vez de uno):
       - finetuned-sd-1.4-oldbookillustrations-entrega-runpod-io https://huggingface.co/Emmy1993/finetuned-sd-1.4-oldbookillustrations-entrega-runpod-io
       - finetuned-sd-1.4-oldbookillustrations-entrega https://huggingface.co/Emmy1993/finetuned-sd-1.4-oldbookillustrations-entrega ( **Defectuoso**)
- He creado dos archivos `.py` para cada finetuneado.
- Finalmente he completado todo el proceso (similar a como lo hicimos en clase).


## 2. Instalación de liberias.
-  En este link: https://huggingface.co/docs/huggingface_hub/v1.21.0.rc0/en/package_reference/authentication#huggingface_hub.notebook_login he encontrado una forma de hacer login a huggingface para poder subir mi modelo. Más adelante veremos como lo hago.

In [ ]:
!pip install -q "diffusers==0.31.0" transformers accelerate datasets torchvision tqdm huggingface_hub

In [ ]:
# Aqui lo utilizo, segun entiendo esta es una forma mas segura de hacer login sin necesidad de exponer el access token ni nada de eso.

from huggingface_hub import notebook_login

notebook_login()

## 3. Importando todas las librerias que utilizaré para este ejercicio.

In [ ]:
import os
import time

from datasets import load_dataset
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
from PIL import ImageFile

## 4. El problema encontrado en mi Macbook Pro M1 de 16 GB de RAM.
Como seguramente usted ya sabe, las mac no vienen con tarjeta gráfica como nvidia o similar, lo que si utilizan es algo que se llama:  **MPS**: Metal Performance Shaders. Inicialmente intenté utilizarlo, pero no funcionaba de hecho reinició la máquina.

In [ ]:
# device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
# Aqui estoy intentarlo utilizarlo, pero no funcionó, Así que intenté usar el CPU pero me estaba marcando un estimado de 10 + horas y se quedó sin memoria varias veces (no entiendo muy bien porque). 
# Al final termine reduciendo el tamaño de las imagenes de 512px a 64px y completó el proceso en 8.5 horas, pero las imagenes no sirvieron porque reducir el tamaño tanto el tunning las hizo irreconocibles, o sea
# no se entiende nada.
# FINALMENTE: Terminé aprendiendo a usar un servicio llamado: https://www.runpod.io/

device = "cuda"

## 4.1 La solución alternativa: https://www.runpod.io/
 - Seguramente ya conoce la herramienta pero básicamente pagas por horas de computo segun el hardware que eligas pagas mas o menos.
 - Elegi una maquina con las siguientes características:
   - GPU A100 PCIe1x
   - Memory 117 GB
   - vCPU 31 (AMD EPYC 7763 64-Core Processor)
- En unos 40 minutos habia completado todo correctamente sin sacrificar las dimensiones de las imagenes (512px) como lo vimos en clase.
- Pagué poco menos de 4 euros.

- Estoy conciente de que tal vez este no era el proposito de esta tarea, pero no tenia opción.

In [ ]:
ImageFile.LOAD_TRUNCATED_IMAGES = True # En un intento de entrenamiento me marcó un error, tuve que investigar y con esto se resuelve el problema.

pretrained_model_name = "CompVis/stable-diffusion-v1-4"

# Cargamos el dataset de ilustraciones de libros antiguos:
dataset_name = "gigant/oldbookillustrations"
max_train_samples = None  # usa el dataset completo (Ya que voy a pagar el servicio queria probar al maximo) (4154 filas); y tiene un tamaño de 5.1 GB de acuerdo a lo que se documenta aqui: https://huggingface.co/datasets/gigant/oldbookillustrations

dataset = load_dataset(dataset_name, split="train")
if max_train_samples:
    dataset = dataset.select(range(max_train_samples))

# Solo nos interesan las columnas "1600px" (imagen) e "info_alt" (descripcion).
# Comprobamos el tamaño de una imagen del dataset (la altura varia entre imagenes):
size = dataset[0]["1600px"].size
print(f"Tamaño de la primera imagen del dataset: {size}")

resolution = 512
image_transforms = transforms.Compose([
    transforms.Resize(resolution),             # redimensiona el lado mas corto a `resolution`
    transforms.CenterCrop(resolution),         # recorta el centro para dejarla cuadrada
    transforms.ToTensor(),                     # convertir a tensor
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # normalizacion
])

# Creamos un Dataset wrapper para la hora del entrenamiento:
# batch_size mayor que en la version de CPU porque una GPU tiene mucha mas memoria
batch_size = 16

class Text2ImageDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        image = image_transforms(example["1600px"].convert("RGB"))
        text = example["info_alt"] or ""
        token = tokenizer(text, padding="max_length", truncation=True, max_length=tokenizer.model_max_length, return_tensors="pt")
        return {
            "pixel_values": image,
            "input_ids": token.input_ids.squeeze(0),
            "attention_mask": token.attention_mask.squeeze(0),
        }


train_dataset = Text2ImageDataset(dataset)
# num_workers para decodificar/redimensionar imagenes en paralelo en vez de bloquear la
# GPU entre batches; pin_memory acelera la copia CPU -> GPU.
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True)

# Tokenizador:
tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name, subfolder="tokenizer")

# Scheduler:
noise_scheduler = DDPMScheduler.from_pretrained(pretrained_model_name, subfolder="scheduler")

# Text Encoder (CLIP):
text_encoder = CLIPTextModel.from_pretrained(
    pretrained_model_name,
    subfolder="text_encoder",
).to(device)

# VAE: Autoencoder:
vae = AutoencoderKL.from_pretrained(
    pretrained_model_name,
    subfolder="vae",
).to(device)

# La UNet:
unet = UNet2DConditionModel.from_pretrained(
    pretrained_model_name,
    subfolder="unet",
).to(device)

# Congelamos los pesos del VAE y del Text Encoder, ya que solo queremos finetunear la UNet:
vae.eval()
text_encoder.eval()

for param in vae.parameters():
    param.requires_grad = False
for param in text_encoder.parameters():
    param.requires_grad = False

# Gradient checkpointing desactivado: con una GPU de 80GB de VRAM (A100) hay mas que
# suficiente memoria sin el, y evitarlo acelera el entrenamiento (no hay que recomputar
# el forward durante el backward). Si usas una GPU con menos VRAM (p. ej. 16-24GB),
# vuelve a activarlo con unet.enable_gradient_checkpointing().

# Optimizador:
learning_rate = 1e-5
optimizer = torch.optim.AdamW(unet.parameters(), lr=learning_rate)

# Acelerador: detecta la GPU automaticamente (sin forzar cpu=True como en la version local).
accelerator = Accelerator()
unet, optimizer, train_dataloader = accelerator.prepare(unet, optimizer, train_dataloader)
print(accelerator.device)

# Checkpoints intermedios: si el proceso se cae (OOM, imagen corrupta, pod interrumpido,
# etc.), al reiniciar el script se reanuda desde el ultimo checkpoint en vez de empezar
# de cero.

# Aqui tambien tuve que hacer esta implementación porque me pasó que al 40% de el primer pod algo falló y tuve que empezar de nuevo. Entonces investigué y al parecer esto me permite ir haciendo 
# saves parciales por si algo sale mal, despues de implementar esto, no falló no tuve tiempo para probarlo completamente.

checkpoint_dir = "./checkpoint-oldbooks"
checkpoint_path = os.path.join(checkpoint_dir, "checkpoint.pt")
checkpoint_every_n_steps = 200

start_epoch = 0
global_step = 0

if os.path.exists(checkpoint_path):
    print(f"Checkpoint encontrado en {checkpoint_path}, reanudando entrenamiento...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    unet.load_state_dict(checkpoint["unet"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    start_epoch = checkpoint["epoch"]
    global_step = checkpoint["global_step"]
    print(f"Reanudando desde la epoca {start_epoch}, global_step {global_step}")


def guardar_checkpoint(epoch_to_resume_at):
    os.makedirs(checkpoint_dir, exist_ok=True)
    torch.save(
        {
            "unet": unet.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch_to_resume_at,
            "global_step": global_step,
        },
        checkpoint_path,
    )


# Training loop:
num_epochs = 2

training_start = time.time()

for epoch in range(start_epoch, num_epochs):
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch}")

    for batch in progress_bar:

        # Se pasan los pixeles al espacio latente con el encoder del VAE:
        with torch.no_grad():
            latents = vae.encode(batch["pixel_values"].to(accelerator.device)).latent_dist.sample()
            latents = latents * 0.18215

        # Proceso de difusion hacia delante:
        # 1. Creamos ruido aleatorio
        noise = torch.randn_like(latents)
        # 2. Cogemos un timestep aleatorio:
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=latents.device).long()
        # 3. Añadimos ruido al vector del espacio latente:
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # Codificamos el texto:
        encoder_hidden_states = text_encoder(batch["input_ids"].to(accelerator.device))[0]

        # Con el vector con ruido, el timestep, y el vector de texto, hacemos la prediccion de ruido:
        noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

        # Calculamos el error y actualizamos los parametros:
        loss = torch.nn.functional.mse_loss(noise_pred, noise)
        accelerator.backward(loss)
        optimizer.step()
        optimizer.zero_grad()

        progress_bar.set_postfix(loss=loss.item())

        global_step += 1
        if global_step % checkpoint_every_n_steps == 0:
            guardar_checkpoint(epoch_to_resume_at=epoch)

    epoch_elapsed = time.time() - training_start
    print(f"Epoca {epoch} completada. Tiempo acumulado: {epoch_elapsed / 3600:.2f} horas")
    guardar_checkpoint(epoch_to_resume_at=epoch + 1)

training_elapsed = time.time() - training_start
print(f"Training loop completado en {training_elapsed / 3600:.2f} horas")

# Guardamos el modelo finetuneado:
output_dir = "./finetuned-model-oldbooks"

unet.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Modelo guardado en {output_dir}")

## 5. Al terminar el entrenamiento en el entorno virtual online de runpod.io
- Descargar el model creado y ponerlo en el directorio con el nombre `finedtuned-model-oldbooks`.
- Para utilizarlo en el otro archivo en el que reviso resultados.